# W5: Your first Replicate API call

**Goal:** connect a prompt to an API call, display the image, and save it.
Open this notebook in Jupyter and run cells in order with **Shift+Enter**.
This introduction has one model and no game controls.

We start in **offline practice mode**. It produces a labelled test diagram,
not an AI-generated image. Switch to live mode with the instructor when ready.
Each live request uses your own Replicate account and may incur a charge.

Initial model: [FLUX.1 schnell](https://replicate.com/black-forest-labs/flux-schnell),
carried over from the previous class guide. The instructor can replace it after checking its inputs.

**Setup, if needed:** run `%pip install "replicate>=1.0.7,<2" "Pillow>=10,<13"`
in a temporary cell, then restart the kernel. The shared setup instructions are in [README.md](README.md).


In [ ]:
%pip install -q replicate Pillow

## 1. Choose practice or live mode

Keep `LIVE = False` for an offline walkthrough. No API token or account request is needed.


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
from uuid import uuid4
import json
from PIL import Image as PILImage, ImageDraw
from IPython.display import display, Image, FileLink

LIVE = False
MODEL = "black-forest-labs/flux-schnell"


## 2. Connect privately

In live mode, get your token from [Replicate](https://replicate.com/account/api-tokens)
and paste it only into the hidden input below. Never paste it into a code cell.
The token stays in kernel memory. Restart the kernel when finished.


In [ ]:
client = None

if LIVE:
    import replicate
    from getpass import getpass

    token = getpass("Replicate API token (hidden): ").strip()
    if not token:
        raise ValueError("Enter your API token, or switch back to practice mode.")
    client = replicate.Client(api_token=token, timeout=120)
    del token
    
    print("Client ready. No generation has been requested.")
else:
    print("Offline practice: no API connection.")


## 3. Write the prompt

This dictionary corresponds to the model's web UI inputs. `prompt` describes the scene;
the other entries request one square PNG. Other models may use different inputs.


In [ ]:
prompt = "A cyclist in a yellow raincoat crosses a foggy stone bridge at dawn, wide shot, muted colours."
inputs = {
    "prompt": prompt,
    "num_outputs": 1,
    "aspect_ratio": "1:1",
    "output_format": "png",
}


## 4. Generate once

The important line is `client.run(MODEL, input=inputs)`.
In live mode, type **GENERATE** to send one paid request. Rerunning this cell can make another request.
Displaying or saving below never generates again.


In [ ]:
# function to generate an image
def generate_image(inputs, client, MODEL):
    model = client.models.get(MODEL)
    version_id = model.latest_version.id

    prediction = client.predictions.create(version=version_id, input=inputs)
    print(prediction.id)

    prediction.wait()
    print(prediction.output)
    
    return prediction.output

In [ ]:
output = None
image_bytes = None
run_context = None
if not prompt.strip():
    raise ValueError("Write a prompt first.")
if LIVE:
    if client is None:
        raise ValueError("Run the connection cell first.")
    if input("One paid image request. Type GENERATE to continue: ") == "GENERATE":
        try:
            output = generate_image(inputs, client, MODEL)
            run_context = {"model": MODEL, "inputs": dict(inputs), "mode": "live"}
            print("Generation returned. Run the next cell to save it.")
        except Exception:
            print("Request failed here. Check token, credit, model inputs and your Replicate predictions before retrying. A prediction may already exist. No automatic retry was made.")
else:
    canvas = PILImage.new("RGB", (640, 400), "#edf2f4")
    draw = ImageDraw.Draw(canvas)
    draw.text((24, 20), "OFFLINE PRACTICE / NOT AI GENERATED", fill="black")
    draw.rectangle((60, 140, 220, 300), fill="#e76f51")
    draw.ellipse((280, 140, 440, 300), fill="#2a9d8f")
    buffer = BytesIO()
    canvas.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    run_context = {"model": None, "inputs": dict(inputs), "mode": "practice"}
    print("Practice diagram ready. It does not respond to your prompt.")
if run_context is not None:
    run_context["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    run_folder = Path("outputs") / "basics" / uuid4().hex[:12]


## 5. Read the result and save it locally

Replicate returns a list of file outputs. `output[0].read()` reads the first file's bytes.
Save promptly rather than relying on its hosted URL. If downloading fails, retry this cell,
not the paid generation cell. Each generation has its own output folder.


In [ ]:
from urllib.request import urlretrieve

if run_context is None:
    print("No result to save. Check the generation cell.")
else:
    try:
        run_folder.mkdir(parents=True, exist_ok=True)
        saved_image = run_folder / "image.png"
        if LIVE and output is not None:
            url = output[0]
            urlretrieve(url, saved_image)
        elif image_bytes is not None:
            with open(saved_image, "wb") as f:
                f.write(image_bytes)
        (run_folder / "prompt.json").write_text(json.dumps(run_context, indent=2), encoding="utf-8")
        print("Saved image and prompt record in", run_folder)
    except Exception:
        saved_image = None
        print("Could not read or save the image. Check the output and local folder permissions. Retry this cell before making another paid request.")


## 6. Display and download

These links download files from your Jupyter server to your computer. No new API call is made.


In [ ]:
if saved_image is not None:
    display(Image(filename=str(saved_image), width=480))
    display(FileLink(str(saved_image)))
    display(FileLink(str(run_folder / "prompt.json")))
else:
    print("Save an image first.")


## Try one change

In live mode, change one detail in the prompt, rerun steps 3 to 6, and compare both saved outputs.
What did you intend to change? What changed unexpectedly? A single pair is an observation,
not proof that a prompt is always better. In practice mode, rehearse saving only; the diagram stays the same.

Next: [Artistic Telephone](02_artistic_telephone.ipynb).
Keep prompt records private during the game. Never submit a token.

Official references: [Python guide](https://replicate.com/docs/get-started/python/),
[model inputs](https://replicate.com/black-forest-labs/flux-schnell/api),
[file outputs](https://replicate.com/docs/topics/predictions/output-files).
